<a href="https://colab.research.google.com/github/At4fn-Yoo/Nefta-Kpx/blob/main/Nefta_Kpx_Deploy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 align="center">Nefta-Kpx HEROKU</h1>

<h3 align="center">Deploy Nefta-Kpx in Heroku using Google Colab</h3>
<hr>

### ***Collab Repository Details***
- 🔗 **Repo :** https://github.com/At4fn-Yoo/Nefta-Kpx
- 🤖 **Version :** _v1.0.0_
- ☢️ **Colab Version :** _v1.5_

In [ ]:
#@title <center><h3>***Heroku Login***</h3></center><br>

#@markdown ---
Heroku_Email = "" #@param {type:"string"}
Heroku_API = "" #@param {type:"string"}
#@markdown ---

!curl -s https://cli-assets.heroku.com/install.sh | sh

from IPython.display import HTML, clear_output, display
clear_output()
display(HTML("<marquee><b>Heroku CLI Installed !</b></marquee>"))

if not all([Heroku_Email, Heroku_API]):
    raise ValueError("Please fill in all Mandatory Variables.")

from os import path as ospath, chmod

netrc_path = ospath.expanduser("~/.netrc")

netrc_creds = f'''machine api.heroku.com
  login {Heroku_Email}
  password {Heroku_API}
machine git.heroku.com
  login {Heroku_Email}
  password {Heroku_API}'''
with open(netrc_path, "w") as netrc_file:
    netrc_file.write(netrc_creds)

chmod(netrc_path, 0o600)

!git config --global user.email {Heroku_Email}
!git config --global user.name "Nefta-Kpx"

display(HTML("<marquee><b>Heroku Email & Password Loaded!</b></marquee>"))

In [ ]:
#@title <center><h3>***Create Heroku Multi App***</h3></center><br>

#@markdown ---
App_Names = "" #@param {type:"string"}
#@markdown <h6>( <b>Syntax:</b> <i>bot_name1 bot_name2, separated by space !</i> )</h6>
#@markdown <h6>( <b>Note:</b> <i>App Name is Optional, skip for random name !</i> )</h6>

Server_Region = "eu" #@param ["eu", "us"] {allow-input: true}
HK_Team_Name = "" #@param {type:"string"}
#@markdown <h6>( <b>Note:</b> <i>Team App Deploy, Optional option only if you want to deploy to Teams !</i> )</h6>

#@markdown ---

HK_Team_Name = f"--team {HK_Team_Name}" if HK_Team_Name else ""
for App_Name in App_Names.split():
    !heroku create --region $Server_Region --stack container $HK_Team_Name $App_Name

In [ ]:
#@title <center><h3>***Nefta-Kpx Repo Config Setup***</h3></center><br>

#@markdown ---
App_Name = "" #@param {type:"string"}
#@markdown <h6>( <b>Note:</b> <i>Config Setup for this App Name, Change the App Name for every Config Save!</i> )</h6>

#@markdown ---

#@markdown #### ***Fill all these Variables for the `config.py`*** **(All are Mandatory)**

BOT_TOKEN = ""  # @param {type:"string"}
TELEGRAM_API = 0  # @param {type:"integer"}
TELEGRAM_HASH = ""  # @param {type:"string"}
OWNER_ID = 0  # @param {type:"integer"}
UPSTREAM_REPO = ""  # @param {type:"string"}
UPSTREAM_BRANCH = "main"  # @param {type:"string"}
DATABASE_URL = ""  # @param {type:"string"}
BASE_URL = ""  # @param {type:"string"}

#@markdown ---

#@markdown ### ***OR***
CONF_GIST_URL = "" # @param {type:"string"}
#@markdown <h6><i>(If you want to Upload `config.py` via `gist.github.com`, Provide the gist URL, Always make Private gist)</i></h6>

#@markdown ---

#@markdown ### ***OR***
Upload_Config = False # @param {type:"boolean"}
#@markdown <h6><i>(If you want to Upload `config.py` file Externally, Tick the Above CheckBox)</i></h6>

#@markdown ---

if not App_Name and not Upload_Config and not CONF_GIST_URL and not all([BOT_TOKEN, TELEGRAM_API, TELEGRAM_HASH, OWNER_ID, UPSTREAM_REPO, DATABASE_URL, BASE_URL]):
    raise ValueError("Please fill in all Mandatory Variables.")

from os import path, remove

if path.isdir(App_Name):
    !rm -rf $App_Name

!git clone https://github.com/At4fn-Yoo/Nefta-Kpx $App_Name
%cd $App_Name

for file in ["README.md"]:
    if path.isfile(file):
        remove(file)

if Upload_Config:
    from google.colab import files
    config_creds = list(files.upload().values())[0]
    with open("config.py", "wb") as config_file:
        config_file.write(config_creds)
    print("config.py File Uploaded and Saved Successfully")
elif CONF_GIST_URL:
    !curl -o "config.py" $CONF_GIST_URL
    print("config.py File Extracted and Saved Successfully")
elif all([BOT_TOKEN, TELEGRAM_API, TELEGRAM_HASH, OWNER_ID, UPSTREAM_REPO, DATABASE_URL, BASE_URL]):
    config_creds = f'BOT_TOKEN = "{BOT_TOKEN}"\n' \
                   f'TELEGRAM_API = {TELEGRAM_API}\n' \
                   f'TELEGRAM_HASH = "{TELEGRAM_HASH}"\n' \
                   f'OWNER_ID = {OWNER_ID}\n' \
                   f'UPSTREAM_REPO = "{UPSTREAM_REPO}"\n' \
                   f'UPSTREAM_BRANCH = "{UPSTREAM_BRANCH}"\n' \
                   f'DATABASE_URL = "{DATABASE_URL}"\n' \
                   f'BASE_URL = "{BASE_URL}"\n' \

    with open("config.py", "wb") as config_file:
        config_file.write(config_creds.encode())
    print("config.py File made and Saved Successfully")

%cd ..

print("All Available Config Bot Names Saved :")
!ls

In [ ]:
# @title <center><h3>***Deploy Heroku Multi App***</h3></center><br>

#@markdown ---
App_Names = "" #@param {type:"string"}
#@markdown <h6>( <b>Syntax:</b> <i>bot_name1 bot_name2, separated by space ! Config must be set for that bot to deploy</i> )</h6>

#@markdown ---

from os import path as ospath

for App_Name in App_Names.split():
    if ospath.isdir(App_Name):
        %cd $App_Name
        !git add . -f
        !git commit -m "HK Setup"
        !heroku git:remote -a $App_Name
        !git push heroku main -f
        %cd ..
    else:
        print(f"Config not set for {App_Name}, Again run the previous cell with App Name")

In [ ]:
# @title <center><h3>***Show Heroku App Logs***</h3></center><br>

#@markdown ---
App_Name = "" #@param {type:"string"}
#@markdown ---

!heroku logs -t -a {App_Name}

In [ ]:
#@title <center><h3>***Heroku Logout***</h3></center><br>

!heroku logout